# 00 语音合成 TTS：小模型拆链路 + VoxCPM2 大模型体验

目标：先用一个较小、Hugging Face 原生支持的模型把 TTS 的基础链路跑清楚，再切到参数更大的 VoxCPM2 看现代语音模型在多语言、音色设计、参考音频克隆和部署成本上的差异。

本教程选择：

| 层级 | 模型 | 适合学习什么 |
| --- | --- | --- |
| 小模型 | `microsoft/speecht5_tts` + `microsoft/speecht5_hifigan` | processor、speaker embedding、声学模型、vocoder、16kHz waveform |
| 大模型 | `openbmb/VoxCPM2` | 2B 参数、48kHz 输出、多语言、voice design、reference audio cloning、CFG 和 diffusion steps |

VoxCPM2 权重较大，本 notebook 默认只运行小模型。要运行 VoxCPM2，把后面的 `RUN_VOXCPM2 = False` 改成 `True`。

## 1. 安装依赖

在仓库根目录运行：

```bash
pip install -r requirements.txt
pip install -r speech/requirements-speech.txt
```

如果访问 Hugging Face 较慢，可以在运行前设置镜像：

```bash
export HF_ENDPOINT=https://hf-mirror.com
```

VoxCPM2 官方文档说明 `pip install voxcpm` 即可使用 Python API；模型加载时会自动选择 `cuda -> mps -> cpu`，也可以显式指定设备。

In [ ]:
from pathlib import Path
import os
import time

import soundfile as sf
import torch

OUTPUT_DIR = Path("speech/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_SPEECHT5 = True
RUN_VOXCPM2 = False

SMALL_TEXT = "Speech synthesis turns written text into a waveform that we can play."
VOXCPM_TEXT = "(年轻女性，温柔自然，语速适中) 你好，这是 VoxCPM2 的语音合成示例。"

print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("mps available:", getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())

## 2. 公共工具：设备选择、参数量、RTF

语音部署里除了“听起来好不好”，还要看两个基础指标：

- `duration`: 生成出来的音频时长。
- `RTF`: real-time factor，生成耗时 / 音频时长。`RTF < 1` 才有机会实时播放。

采样率也很关键：16kHz 文件小、速度快；48kHz 更适合高保真输出，但文件和后处理成本更高。

In [ ]:
def pick_torch_device():
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters())


def save_wav(path, waveform, sample_rate, elapsed_seconds):
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(path, waveform, sample_rate)
    duration_seconds = len(waveform) / sample_rate
    rtf = elapsed_seconds / duration_seconds if duration_seconds else float("inf")
    print(f"saved: {path}")
    print(f"sample_rate={sample_rate}, duration={duration_seconds:.2f}s, elapsed={elapsed_seconds:.2f}s, rtf={rtf:.2f}")
    return {"path": str(path), "sample_rate": sample_rate, "duration_seconds": duration_seconds, "elapsed_seconds": elapsed_seconds, "rtf": rtf}

## 3. 小模型：SpeechT5 的 TTS 链路

SpeechT5 这条链路很适合教学，因为模块边界清楚：

```text
text
-> SpeechT5Processor 变成 input_ids
-> speaker embedding 指定说话人特征
-> SpeechT5ForTextToSpeech 生成声学表示
-> SpeechT5HifiGan vocoder 生成 waveform
-> 保存 16kHz WAV
```

注意：这个 checkpoint 主要面向英文 TTS。中文、多语言和音色控制放到 VoxCPM2 章节看。

In [ ]:
if RUN_SPEECHT5:
    from datasets import load_dataset
    from transformers import SpeechT5ForTextToSpeech, SpeechT5HifiGan, SpeechT5Processor

    device = pick_torch_device()
    print("device:", device)

    processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
    speecht5 = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts").to(device)
    vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan").to(device)

    embeddings_dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
    speaker_embeddings = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0).to(device)

    inputs = processor(text=SMALL_TEXT, return_tensors="pt")
    input_ids = inputs["input_ids"].to(device)

    print("input_ids shape:", tuple(input_ids.shape))
    print("speaker_embeddings shape:", tuple(speaker_embeddings.shape))
    print(f"SpeechT5 acoustic model params: {count_parameters(speecht5) / 1e6:.1f}M")
    print(f"HiFi-GAN vocoder params: {count_parameters(vocoder) / 1e6:.1f}M")

In [ ]:
if RUN_SPEECHT5:
    start = time.perf_counter()
    with torch.inference_mode():
        speech = speecht5.generate_speech(input_ids, speaker_embeddings, vocoder=vocoder)
    elapsed = time.perf_counter() - start

    speecht5_metrics = save_wav(
        OUTPUT_DIR / "speecht5_demo.wav",
        speech.detach().cpu().numpy(),
        16_000,
        elapsed,
    )
    speecht5_metrics

## 4. 从小模型里观察部署问题

小模型已经能暴露 TTS 部署的核心问题：

- 文本前处理会影响读法，比如数字、缩写、标点和多语言文本。
- speaker embedding 或 reference audio 是音色条件，生产里要管理来源、授权和质量。
- vocoder / decoder 是波形质量和速度的重要瓶颈。
- 输出 WAV 之前要明确采样率，否则播放速度、音调和下游处理都会出错。
- RTF 可以把体验和算力联系起来：同一段文本，模型越慢、音频越长，服务并发越难做。

In [ ]:
if RUN_SPEECHT5:
    decoded_tokens = processor.tokenizer.convert_ids_to_tokens(input_ids[0].detach().cpu().tolist())
    print("first tokens:", decoded_tokens[:30])
    print("token count:", len(decoded_tokens))
    print("output samples:", len(speech))
    print("waveform min/max:", float(speech.min()), float(speech.max()))

## 5. 大模型：VoxCPM2 voice design

VoxCPM2 是更大的现代 TTS 模型。官方文档给出的关键信息包括：2B 参数、48kHz 输出、30 种语言、voice design、style control、reference audio cloning，以及 `generate()` API。

最小代码结构是：

```python
from voxcpm import VoxCPM
model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)
wav = model.generate(text="...", cfg_value=2.0, inference_timesteps=10)
```

本教程里用 `optimize=False` 保守运行，减少部分 CPU/MPS 环境里 `torch.compile` 的兼容性变量；CUDA 环境可以自行改成默认优化。

In [ ]:
if RUN_VOXCPM2:
    from voxcpm import VoxCPM

    print("Loading VoxCPM2. First run downloads large model weights.")
    voxcpm2 = VoxCPM.from_pretrained(
        "openbmb/VoxCPM2",
        device="auto",
        load_denoiser=False,
        optimize=False,
    )

    start = time.perf_counter()
    wav = voxcpm2.generate(
        text=VOXCPM_TEXT,
        cfg_value=2.0,
        inference_timesteps=10,
        normalize=True,
    )
    elapsed = time.perf_counter() - start

    voxcpm2_metrics = save_wav(
        OUTPUT_DIR / "voxcpm2_voice_design.wav",
        wav,
        voxcpm2.tts_model.sample_rate,
        elapsed,
    )
    voxcpm2_metrics
else:
    print("Skip VoxCPM2. Change RUN_VOXCPM2 to True when you have time and disk/GPU budget.")

## 6. VoxCPM2 参考音频克隆

如果你有一段干净的参考音频，可以用 `reference_wav_path`。VoxCPM2 的参考音频克隆不要求提供参考音频的逐字 transcript；参考音频主要控制“谁在说”，括号里的文本控制语气、速度和风格。

参考音频建议：

- 5 到 30 秒。
- 尽量干净、少混响、少背景噪声。
- 说话人授权明确，不要克隆没有授权的真实人物声音。

In [ ]:
REFERENCE_WAV_PATH = "speech/reference_speaker.wav"

if RUN_VOXCPM2 and Path(REFERENCE_WAV_PATH).exists():
    start = time.perf_counter()
    cloned = voxcpm2.generate(
        text="(语气更轻松，稍微慢一点) 这是一段参考音频克隆的测试。",
        reference_wav_path=REFERENCE_WAV_PATH,
        cfg_value=2.0,
        inference_timesteps=10,
        normalize=True,
    )
    elapsed = time.perf_counter() - start
    save_wav(
        OUTPUT_DIR / "voxcpm2_reference_clone.wav",
        cloned,
        voxcpm2.tts_model.sample_rate,
        elapsed,
    )
elif RUN_VOXCPM2:
    print(f"No reference file found: {REFERENCE_WAV_PATH}")
else:
    print("Reference cloning section is skipped because RUN_VOXCPM2 is False.")

## 7. 小模型和大模型怎么对比

| 维度 | SpeechT5 | VoxCPM2 |
| --- | --- | --- |
| 教学价值 | 模块边界清楚，适合拆 processor/model/vocoder | 更接近现代 TTS 产品能力 |
| 输出采样率 | 16kHz | 48kHz |
| 多语言 | 主要看英文 checkpoint | 支持多语言语音合成 |
| 音色条件 | speaker embedding | voice design、reference audio、prompt audio |
| 部署成本 | 较低 | 高，模型权重、冷启动和显存压力更明显 |
| 适合场景 | 入门、链路学习、轻量实验 | 高质量合成、克隆、风格控制、多语言验证 |

面试或项目里可以这样讲：小模型负责让我理解 TTS 的基础结构，大模型负责验证真实产品里的质量、控制、成本和稳定性问题。

## 8. 部署检查清单

上线 TTS 服务前，至少检查：

- 输入：文本规范化、语言识别、敏感内容、长文本切分。
- 音色：reference audio 授权、质量、时长、去噪策略。
- 推理：设备、dtype、冷启动、RTF、并发、队列超时。
- 输出：采样率、响度、静音裁剪、文件格式、流式播放。
- 稳定性：长文本漂移、生成过短/过长、噪声、重复、异常重试。
- 成本：GPU 显存、模型下载体积、缓存目录、请求峰值和降级模型。

## 参考资料

- SpeechT5 model card: https://huggingface.co/microsoft/speecht5_tts
- VoxCPM2 Quick Start: https://voxcpm.readthedocs.io/en/latest/quickstart.html
- VoxCPM2 Usage Guide: https://voxcpm.readthedocs.io/en/latest/usage_guide.html
- VoxCPM2 model overview: https://voxcpm.readthedocs.io/en/latest/models/voxcpm2.html